# Does Unemployment Increase During Recessions?

This notebook tests whether the US unemployment rate is higher, and rises faster, during NBER-dated recessions than during expansions.

**Expectation:** Yes. During economic downturns, firms hire less and lay off more workers, so unemployment should rise.

The analysis covers January 2000 to the present, using monthly data from FRED.

## Setup

The libraries used in this notebook:

- **fredapi** pulls series directly from the FRED API.
- **python-dotenv** loads the FRED API key from a local `.env` file. The file is listed in `.gitignore`, so the key never enters the repo.
- **pandas** cleans and merges the data.
- **matplotlib** draws the charts.
- **statsmodels** runs the regressions.

In [1]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf
from dotenv import load_dotenv
from fredapi import Fred

## Data Pull

Two monthly series come from FRED:

| Series ID | Description | Original source |
|---|---|---|
| `UNRATE` | Civilian unemployment rate (%) | Bureau of Labor Statistics |
| `USREC` | Recession indicator (1 = recession month, 0 = not) | National Bureau of Economic Research |

The pull starts at 1999-12-01, one month before the analysis window. That extra month means the monthly change in unemployment can be computed for January 2000.

In [2]:
# Read the key from .env rather than hard-coding it, so it stays out of the repo.
load_dotenv()
api_key = os.getenv("FRED_API_KEY")
if not api_key:
    raise RuntimeError("FRED_API_KEY not found; add it to the .env file in this folder.")

fred = Fred(api_key=api_key)

START = "1999-12-01"
unrate = fred.get_series("UNRATE", observation_start=START).rename("UNRATE")
usrec = fred.get_series("USREC", observation_start=START).rename("USREC")

In [3]:
display(unrate.head())
display(usrec.head())

1999-12-01    4.0
2000-01-01    4.0
2000-02-01    4.1
2000-03-01    4.0
2000-04-01    3.8
Name: UNRATE, dtype: float64

1999-12-01    0.0
2000-01-01    0.0
2000-02-01    0.0
2000-03-01    0.0
2000-04-01    0.0
Name: USREC, dtype: float64